# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [12]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [16]:
# TODO: Import the necessary libs
# For example: 
import os
from typing import Any, Dict, List, Optional

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import (
    UserMessage,
    SystemMessage,
    ToolMessage,
    AIMessage,
)
from lib.tooling import tool

In [17]:
from dotenv import load_dotenv

# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

import chromadb
from lib.tooling import tool

# Connect to the existing persistent database
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")


@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds the most relevant results in the vector database.

    Args:
        query: A question about the video game industry.

    Returns:
        A list of relevant game records containing Platform, Name,
        YearOfRelease, and Description.
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
        include=["metadatas", "documents", "distances"],
    )

    games = []

    if not results["metadatas"] or not results["metadatas"][0]:
        return games

    for metadata in results["metadatas"][0]:
        games.append({
            "Platform": metadata.get("Platform", "Unknown"),
            "Name": metadata.get("Name", "Unknown"),
            "YearOfRelease": metadata.get("YearOfRelease", "Unknown"),
            "Description": metadata.get("Description", ""),
        })

    return games

#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
from pydantic import BaseModel, Field
from lib.llm import LLM
from lib.tooling import tool


class EvaluationReport(BaseModel):
    useful: bool = Field(
        description="Whether the retrieved documents are sufficient to answer the question."
    )
    description: str = Field(
        description="Detailed explanation of the retrieval evaluation and any missing information."
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> EvaluationReport:
    """
    Based on the user's question and the list of retrieved documents,
    analyze whether the documents are useful for answering the question.

    Args:
        question: Original question from the user.
        retrieved_docs: Documents retrieved from the vector database.

    Returns:
        EvaluationReport containing:
        - useful: Whether the documents are useful to answer the question.
        - description: Detailed explanation of the evaluation result.
    """
    llm = LLM()

    prompt = f"""
    Your task is to evaluate whether the retrieved documents are sufficient
    to answer the user's question.

    User question:
    {question}

    Retrieved documents:
    {retrieved_docs}

    Evaluate the documents based on:
    1. Relevance to the user's question.
    2. Coverage of the information needed to answer it.
    3. Whether the documents provide enough evidence for a reliable answer.
    4. What information is missing, if any.

    Set useful to true only if the documents provide sufficient relevant
    information to answer the question. Otherwise, set it to false.

    Give a detailed explanation so that another agent can decide whether
    to accept the retrieved documents or retrieve more information.

    Return the result in the EvaluationReport format:
    - useful: boolean
    - description: string
    """

    # TODO: Call your LLM and parse its response as EvaluationReport.
    # Replace this line with the structured-output method supported by lib/llm.py.
    response = llm.invoke(prompt, response_format=EvaluationReport)

    return response

#### Game Web Search Tool

In [6]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry.

# TODO: Create game_web_search tool

import os
from tavily import TavilyClient
from lib.tooling import tool


@tool
def game_web_search(question: str) -> str:
    """
    Semantic search: Finds relevant information about the game industry
    on the web.

    Args:
        question: A question about the game industry.

    Returns:
        Search results containing relevant information from the web.
    """
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

    response = client.search(
        query=question,
        search_depth="advanced",
        topic="general",
        max_results=5,
    )

    results = response.get("results", [])

    if not results:
        return "No relevant web search results were found."

    formatted_results = []

    for result in results:
        formatted_results.append(
            f"Title: {result.get('title', 'N/A')}\n"
            f"URL: {result.get('url', 'N/A')}\n"
            f"Content: {result.get('content', 'N/A')}"
        )

    return "\n\n".join(formatted_results)

### Agent

In [8]:

from lib.agents import Agent

# 1. Define the instructions
game_instructions = """
You are an expert assistant specializing in the video game industry.

Answer questions accurately using reliable evidence.
Use game_web_search to find current information about games.
Use evaluate_retrieval to assess retrieved documents when available.
Never invent facts or sources.
Include source URLs when available.
Keep answers clear, concise, and readable.
"""

# 2. Register the tools
game_tools = [
    evaluate_retrieval,
    game_web_search,
]

# 3. Create the agent
game_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=game_instructions,
    tools=game_tools,
    temperature=0.2,
)

print("Game agent created successfully!")


Game agent created successfully!


In [9]:
from contextlib import redirect_stdout
from io import StringIO
from lib.messages import AIMessage

def ask_game_agent(question: str) -> str:
    # Hide StateMachine debug output
    with redirect_stdout(StringIO()):
        run = game_agent.invoke(question)

    final_state = run.get_final_state()
    messages = final_state.get("messages", [])

    for message in reversed(messages):
        content = getattr(message, "content", None)
        if isinstance(message, AIMessage) and isinstance(content, str) and content.strip():
            return content.strip()

    return "Sorry, no final answer was found. Please check the agent's messages."


questions = [
    "When were Pokémon Gold and Silver released?",
    "Which was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for i, question in enumerate(questions, start=1):
    print(f"\n{'=' * 60}")
    print(f"Question {i}: {question}\n")
    print(ask_game_agent(question))

print("\n" + "=" * 60)



Question 1: When were Pokémon Gold and Silver released?

Pokémon Gold and Silver were released in Japan on November 21, 1999. They were later released in North America on October 16, 2000, and in Europe on April 6, 2001.

Question 2: Which was the first 3D platformer Mario game?

The first 3D platformer Mario game is "Super Mario 64," which was released for the Nintendo 64 on June 23, 1996, in Japan and later in North America on September 29, 1996. It was groundbreaking for its use of 3D graphics and open-world gameplay, setting a new standard for platforming games.

Question 3: Was Mortal Kombat X released for PlayStation 5?

Mortal Kombat X was not specifically released for the PlayStation 5; however, it is playable on the PS5 due to backward compatibility with PlayStation 4 games. It was originally released for the PS4 on April 14, 2015. Additionally, Mortal Kombat X is included in the PlayStation Plus Collection for PS5 subscribers, allowing players to access it on the new console

### (Optional) Advanced

In [36]:
# TODO: Update your agent with long-term memory
import sqlite3
from lib.messages import UserMessage, AIMessage

memory_db = "udaplay_memory.db"


def init_memory():
    with sqlite3.connect(memory_db) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS conversation_memory (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                role TEXT NOT NULL,
                content TEXT NOT NULL
            )
        """)


def save_memory(session_id, role, content):
    with sqlite3.connect(memory_db) as conn:
        conn.execute(
            """INSERT INTO conversation_memory
               (session_id, role, content)
               VALUES (?, ?, ?)""",
            (session_id, role, content),
        )


def load_memory(session_id, limit=10):
    with sqlite3.connect(memory_db) as conn:
        rows = conn.execute(
            """SELECT role, content FROM conversation_memory
               WHERE session_id = ?
               ORDER BY id DESC LIMIT ?""",
            (session_id, limit),
        ).fetchall()

    return list(reversed(rows))


init_memory()

In [35]:
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

from typing import TypedDict, Any
from langgraph.graph import StateGraph, START, END


# State shared between all nodes
class GameAgentState(TypedDict, total=False):
    query: str
    retrieved_games: list
    evaluation: dict
    web_results: list
    answer: str


def build_agent(
    retrieve_game,
    evaluate_retrieval,
    game_web_search,
    generate_answer,
):
    # Node 1: Retrieve games from local ChromaDB
    def retrieve_node(state: GameAgentState):
        results = retrieve_game(state["query"])
        return {"retrieved_games": results}

    # Node 2: Evaluate retrieval quality
    def evaluate_node(state: GameAgentState):
        evaluation = evaluate_retrieval({
            "query": state["query"],
            "results": state.get("retrieved_games", []),
        })
        return {"evaluation": evaluation}

    # Node 3: Search the web only if local evidence is insufficient
    def web_search_node(state: GameAgentState):
        results = game_web_search(state["query"])
        return {"web_results": results}

    # Node 4: Generate the final answer
    def answer_node(state: GameAgentState):
        answer = generate_answer(
            query=state["query"],
            games=state.get("retrieved_games", []),
            web_results=state.get("web_results", []),
        )
        return {"answer": answer}

    # Conditional routing after evaluation
    def route_after_evaluation(state: GameAgentState):
        evaluation = state.get("evaluation", {})
        if evaluation.get("sufficient", False):
            return "answer"
        return "web_search"

    # Define the graph
    graph = StateGraph(GameAgentState)

    graph.add_node("retrieve_game", retrieve_node)
    graph.add_node("evaluate_retrieval", evaluate_node)
    graph.add_node("game_web_search", web_search_node)
    graph.add_node("generate_answer", answer_node)

    graph.add_edge(START, "retrieve_game")
    graph.add_edge("retrieve_game", "evaluate_retrieval")

    graph.add_conditional_edges(
        "evaluate_retrieval",
        route_after_evaluation,
        {
            "answer": "generate_answer",
            "web_search": "game_web_search",
        },
    )

    graph.add_edge("game_web_search", "generate_answer")
    graph.add_edge("generate_answer", END)

    return graph.compile()


## TESTS FOR EACH MODULE

### Notebook 2 Setup Validation Test

In [19]:
import importlib.util
import sys
import sqlite3

print("UdaPlay — Notebook 2 Setup Validation")
print("-" * 40)

checks_passed = 0
checks_total = 0

def check(name, passed, details=""):
    global checks_passed, checks_total
    checks_total += 1

    if passed:
        checks_passed += 1
        print(f"✓ PASS: {name}" + (f" — {details}" if details else ""))
    else:
        print(f"✗ FAIL: {name}" + (f" — {details}" if details else ""))

# 1. Check pysqlite3 availability
try:
    if importlib.util.find_spec("pysqlite3") is not None:
        import pysqlite3
        sys.modules["sqlite3"] = pysqlite3
        check("pysqlite3 compatibility", True, "Using pysqlite3")
    else:
        check(
            "pysqlite3 compatibility",
            True,
            "Not installed; using standard sqlite3"
        )
except Exception as e:
    check("pysqlite3 compatibility", False, str(e))

# 2. Verify SQLite works
try:
    import sqlite3
    with sqlite3.connect(":memory:") as connection:
        result = connection.execute("SELECT 1").fetchone()
        assert result == (1,)
    check("SQLite database", True, "Test query successful")
except Exception as e:
    check("SQLite database", False, str(e))

# 3. Check ChromaDB import
try:
    import chromadb
    check("ChromaDB import", True, "Available")
except Exception as e:
    check("ChromaDB import", False, str(e))

# 4. Check embedding functions
try:
    from chromadb.utils import embedding_functions
    check("ChromaDB embedding functions", True, "Available")
except Exception as e:
    check("ChromaDB embedding functions", False, str(e))

print("-" * 40)
print(f"Result: {checks_passed}/{checks_total} checks passed")

if checks_passed == checks_total:
    print("✓ PASS: Notebook 2 setup is ready!")
else:
    print(f"✗ FAIL: {checks_total - checks_passed} check(s) need attention.")


UdaPlay — Notebook 2 Setup Validation
----------------------------------------
✓ PASS: pysqlite3 compatibility — Using pysqlite3
✓ PASS: SQLite database — Test query successful
✓ PASS: ChromaDB import — Available
✓ PASS: ChromaDB embedding functions — Available
----------------------------------------
Result: 4/4 checks passed
✓ PASS: Notebook 2 setup is ready!


### Agent Libraries Setup Test

In [20]:
import os
import sys
import importlib

print("UdaPlay — Agent Libraries Setup Validation")
print("-" * 45)

checks_passed = 0
checks_total = 0

def check(name, passed, details=""):
    global checks_passed, checks_total
    checks_total += 1

    if passed:
        checks_passed += 1
        print(f"✓ PASS: {name}" + (f" — {details}" if details else ""))
    else:
        print(f"✗ FAIL: {name}" + (f" — {details}" if details else ""))

# 1. Check Python can find the custom lib package
try:
    import lib
    check("Import lib package", True, getattr(lib, "__file__", "Available"))
except Exception as e:
    check("Import lib package", False, str(e))

# 2. Check each custom module
modules = [
    "lib.agents",
    "lib.llm",
    "lib.messages",
    "lib.tooling",
]

for module_name in modules:
    try:
        module = importlib.import_module(module_name)
        check(f"Import {module_name}", True)
    except Exception as e:
        check(f"Import {module_name}", False, str(e))

# 3. Check the required classes and message types
required_objects = {
    "lib.agents": ["Agent"],
    "lib.llm": ["LLM"],
    "lib.messages": [
        "UserMessage",
        "SystemMessage",
        "ToolMessage",
        "AIMessage",
    ],
    "lib.tooling": ["tool"],
}

for module_name, object_names in required_objects.items():
    try:
        module = importlib.import_module(module_name)
        for object_name in object_names:
            check(
                f"{module_name}.{object_name}",
                hasattr(module, object_name),
                "Available" if hasattr(module, object_name)
                else "Not found"
            )
    except Exception as e:
        for object_name in object_names:
            check(f"{module_name}.{object_name}", False, str(e))

print("-" * 45)
print(f"Result: {checks_passed}/{checks_total} checks passed")

if checks_passed == checks_total:
    print("✓ PASS: Agent libraries are ready!")
else:
    print(f"✗ FAIL: {checks_total - checks_passed} check(s) need attention.")


UdaPlay — Agent Libraries Setup Validation
---------------------------------------------
✓ PASS: Import lib package — C:\Users\Savaric\PycharmProjects\UdaPlay\lib\__init__.py
✓ PASS: Import lib.agents
✓ PASS: Import lib.llm
✓ PASS: Import lib.messages
✓ PASS: Import lib.tooling
✓ PASS: lib.agents.Agent — Available
✓ PASS: lib.llm.LLM — Available
✓ PASS: lib.messages.UserMessage — Available
✓ PASS: lib.messages.SystemMessage — Available
✓ PASS: lib.messages.ToolMessage — Available
✓ PASS: lib.messages.AIMessage — Available
✓ PASS: lib.tooling.tool — Available
---------------------------------------------
Result: 12/12 checks passed
✓ PASS: Agent libraries are ready!


### Environment Variables Setup Test

In [21]:
import os
from dotenv import load_dotenv

print("UdaPlay — Environment Variables Validation")
print("-" * 45)

checks_passed = 0
checks_total = 0

def check(name, passed, details=""):
    global checks_passed, checks_total
    checks_total += 1

    if passed:
        checks_passed += 1
        print(f"✓ PASS: {name}" + (f" — {details}" if details else ""))
    else:
        print(f"✗ FAIL: {name}" + (f" — {details}" if details else ""))

# 1. Load environment variables
try:
    env_loaded = load_dotenv()
    check(
        ".env loading",
        env_loaded or os.path.isfile(".env"),
        "Environment loaded or .env file found"
        if env_loaded or os.path.isfile(".env")
        else ".env file not found in the current directory"
    )
except Exception as e:
    check(".env loading", False, str(e))

# 2. Read API keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

check(
    "OPENAI_API_KEY",
    bool(OPENAI_API_KEY),
    "Configured" if OPENAI_API_KEY else "Missing or not loaded"
)

check(
    "TAVILY_API_KEY",
    bool(TAVILY_API_KEY),
    "Configured" if TAVILY_API_KEY else "Missing or not loaded"
)

print("-" * 45)
print(f"Result: {checks_passed}/{checks_total} checks passed")

if checks_passed == checks_total:
    print("✓ PASS: Environment setup is ready!")
else:
    print(f"✗ FAIL: {checks_total - checks_passed} check(s) need attention.")


UdaPlay — Environment Variables Validation
---------------------------------------------
✓ PASS: .env loading — Environment loaded or .env file found
✓ PASS: OPENAI_API_KEY — Configured
✓ PASS: TAVILY_API_KEY — Configured
---------------------------------------------
Result: 3/3 checks passed
✓ PASS: Environment setup is ready!


### retrieve_game Tool Validation Test

In [22]:
print("UdaPlay — retrieve_game Validation")
print("-" * 45)

checks_passed = 0
checks_total = 0

def check(name, passed, details=""):
    global checks_passed, checks_total
    checks_total += 1

    if passed:
        checks_passed += 1
        print(f"✓ PASS: {name}" + (f" — {details}" if details else ""))
    else:
        print(f"✗ FAIL: {name}" + (f" — {details}" if details else ""))

# 1. Check collection connection
try:
    count = collection.count()
    check("Vector DB connection", True, f"{count} documents indexed")
except Exception as e:
    check("Vector DB connection", False, str(e))

# 2. Check tool definition
check(
    "retrieve_game tool defined",
    callable(retrieve_game),
    "Tool is callable" if callable(retrieve_game) else "Tool is not callable"
)

# 3. Test semantic retrieval
try:
    results = retrieve_game("video games")

    check(
        "Semantic search",
        isinstance(results, list) and len(results) > 0,
        f"Returned {len(results)} game(s)"
        if isinstance(results, list)
        else "Expected a list"
    )

    # 4. Validate result structure
    required_fields = [
        "Platform", "Name", "YearOfRelease", "Description"
    ]

    valid_records = (
        isinstance(results, list)
        and all(
            isinstance(game, dict)
            and all(field in game for field in required_fields)
            for game in results
        )
    )

    check(
        "Game record structure",
        valid_records,
        "All required fields are present"
        if valid_records else "One or more records have missing fields"
    )

    # Show one example for the reviewer
    if results:
        print("\nExample retrieved game:")
        print(f"  Name: {results[0]['Name']}")
        print(f"  Platform: {results[0]['Platform']}")
        print(f"  Year: {results[0]['YearOfRelease']}")
        print(f"  Description: {str(results[0]['Description'])[:150]}")

except Exception as e:
    check("Semantic search", False, f"{type(e).__name__}: {e}")
    check("Game record structure", False, "Retrieval failed")

print("\n" + "-" * 45)
print(f"Result: {checks_passed}/{checks_total} checks passed")

if checks_passed == checks_total:
    print("✓ PASS: retrieve_game is working!")
else:
    print(f"✗ FAIL: {checks_total - checks_passed} check(s) need attention.")


UdaPlay — retrieve_game Validation
---------------------------------------------
✓ PASS: Vector DB connection — 15 documents indexed
✓ PASS: retrieve_game tool defined — Tool is callable
✓ PASS: Semantic search — Returned 5 game(s)
✓ PASS: Game record structure — All required fields are present

Example retrieved game:
  Name: Wii Sports
  Platform: Wii
  Year: 2006
  Description: A collection of sports games that utilize the Wii's motion controls, bundled with the console to showcase its capabilities.

---------------------------------------------
Result: 4/4 checks passed
✓ PASS: retrieve_game is working!


### game_web_search Tool Validation Test

In [23]:
import os

print("UdaPlay — game_web_search Validation")
print("-" * 45)

checks_passed = 0
checks_total = 0

def check(name, passed, details=""):
    global checks_passed, checks_total
    checks_total += 1

    if passed:
        checks_passed += 1
        print(f"✓ PASS: {name}" + (f" — {details}" if details else ""))
    else:
        print(f"✗ FAIL: {name}" + (f" — {details}" if details else ""))

# 1. Check Tavily API key
api_key_available = bool(os.getenv("TAVILY_API_KEY"))
check(
    "Tavily API key",
    api_key_available,
    "Configured" if api_key_available else "Missing or not loaded"
)

# 2. Check tool definition
check(
    "game_web_search tool defined",
    callable(game_web_search),
    "Tool is callable" if callable(game_web_search) else "Tool is not callable"
)

# 3. Run a real web search, if the key is available
if api_key_available:
    try:
        response = game_web_search(
            "Recent trends in the video game industry"
        )

        has_results = (
            isinstance(response, str)
            and len(response.strip()) > 0
            and "No relevant web search results were found." not in response
        )

        check(
            "Tavily web search",
            has_results,
            "Search returned content" if has_results
            else "No relevant results returned"
        )

        if has_results:
            print("\nExample search output:")
            print(response[:600] + ("..." if len(response) > 600 else ""))

    except Exception as e:
        check(
            "Tavily web search",
            False,
            f"{type(e).__name__}: {e}"
        )
else:
    print("⚠️ Web search skipped because TAVILY_API_KEY is missing.")

print("\n" + "-" * 45)
print(f"Result: {checks_passed}/{checks_total} checks passed")

if checks_passed == checks_total:
    print("✓ PASS: game_web_search is working!")
else:
    print(f"✗ FAIL: {checks_total - checks_passed} check(s) need attention.")


UdaPlay — game_web_search Validation
---------------------------------------------
✓ PASS: Tavily API key — Configured
✓ PASS: game_web_search tool defined — Tool is callable
✓ PASS: Tavily web search — Search returned content

Example search output:
Title: Top Video Game Industry Trends for 2026 | Juego Studios
URL: https://www.juegostudio.com/blog/recent-trends-reforming-gaming-industry
Content: Key policy shifts:

 AI-generated asset labeling becomes legally required
 Loot box restrictions expand across Europe and emerging markets
 Fair-play certifications increase visibility in app stores
 Wellbeing tools—like fatigue detection or break prompts—become standard for live-service titles

These guardrails protect players while helping studios build sustainable long-term communities.

## Conclusion

The most significant video game industry ...

---------------------------------------------
Result: 3/3 checks passed
✓ PASS: game_web_search is working!


### EndToEnd Agent Validation Test

In [24]:
from contextlib import redirect_stdout
from io import StringIO

print("UdaPlay — Game Agent Validation")
print("-" * 50)

checks_passed = 0
checks_total = 0

def check(name, passed, details=""):
    global checks_passed, checks_total
    checks_total += 1

    if passed:
        checks_passed += 1
        print(f"✓ PASS: {name}" + (f" — {details}" if details else ""))
    else:
        print(f"✗ FAIL: {name}" + (f" — {details}" if details else ""))

# Test questions
questions = [
    "When were Pokémon Gold and Silver released?",
    "Which was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for i, question in enumerate(questions, start=1):
    try:
        # Suppress agent debug output during testing
        with redirect_stdout(StringIO()):
            answer = ask_game_agent(question)

        valid_answer = (
            isinstance(answer, str)
            and bool(answer.strip())
            and "Sorry, no final answer was found." not in answer
        )

        check(
            f"Agent question {i}",
            valid_answer,
            "Received a non-empty answer" if valid_answer
            else "No usable answer returned"
        )

        print(f"\nQuestion: {question}")
        print(f"Answer: {answer[:400]}" +
              ("..." if len(answer) > 400 else ""))

    except Exception as e:
        check(
            f"Agent question {i}",
            False,
            f"{type(e).__name__}: {e}"
        )

print("\n" + "-" * 50)
print(f"Result: {checks_passed}/{checks_total} checks passed")

if checks_passed == checks_total:
    print("✓ PASS: Agent end-to-end tests completed!")
else:
    print(f"✗ FAIL: {checks_total - checks_passed} test(s) need attention.")


UdaPlay — Game Agent Validation
--------------------------------------------------
✓ PASS: Agent question 1 — Received a non-empty answer

Question: When were Pokémon Gold and Silver released?
Answer: Pokémon Gold and Silver were released on the following dates:

- Japan: November 21, 1999
- North America: October 16, 2000
- Europe: April 6, 2001
✓ PASS: Agent question 2 — Received a non-empty answer

Question: Which was the first 3D platformer Mario game?
Answer: The first 3D platformer Mario game is "Super Mario 64," which was released for the Nintendo 64 on June 23, 1996, in Japan and later on September 29, 1996, in North America. This game was revolutionary for its use of 3D graphics and open-world gameplay, setting a new standard for platforming games.
✓ PASS: Agent question 3 — Received a non-empty answer

Question: Was Mortal Kombat X released for PlayStation 5?
Answer: Mortal Kombat X was not specifically released for the PlayStation 5; however, it is playable on the PS5 due to

### (Optional) Advanced Tests

In [39]:
import os
import uuid
import sqlite3
import traceback

print("UdaPlay — Advanced Options Validation")
print("=" * 55)

checks_passed = 0
checks_total = 0


def check(name, passed, details=""):
    global checks_passed, checks_total
    checks_total += 1

    if passed:
        checks_passed += 1
        status = "✓ PASS"
    else:
        status = "✗ FAIL"

    print(f"{status}: {name}" + (f" — {details}" if details else ""))


# ==================================================
# 1. Long-term memory
# ==================================================
print("\n[1] LONG-TERM MEMORY")
print("-" * 55)

memory_ready = all(
    callable(globals().get(name))
    for name in ["init_memory", "save_memory", "load_memory"]
)

check(
    "Memory functions available",
    memory_ready,
    "init_memory, save_memory and load_memory"
    if memory_ready else "Define the memory functions first"
)

if memory_ready:
    test_session = f"test_{uuid.uuid4().hex}"

    try:
        init_memory()

        save_memory(test_session, "user", "I like adventure games.")
        save_memory(test_session, "assistant", "I'll keep that in mind.")

        history = load_memory(test_session)

        memory_ok = (
            len(history) == 2
            and history[0][0] == "user"
            and history[0][1] == "I like adventure games."
            and history[1][0] == "assistant"
        )

        check(
            "Memory save and load",
            memory_ok,
            f"Retrieved {len(history)} test messages"
        )

    except Exception as e:
        check("Memory save and load", False, f"{type(e).__name__}: {e}")

    finally:
        # Delete only records belonging to this test session
        try:
            with sqlite3.connect(memory_db) as conn:
                conn.execute(
                    "DELETE FROM conversation_memory WHERE session_id = ?",
                    (test_session,)
                )
        except Exception:
            pass


# ==================================================
# 2. Retrieval tool
# ==================================================
print("\n[2] LOCAL GAME RETRIEVAL")
print("-" * 55)

retrieve_fn = globals().get("retrieve_game")

check(
    "retrieve_game available",
    callable(retrieve_fn),
    "Tool is defined" if callable(retrieve_fn) else "Missing tool"
)

if callable(retrieve_fn):
    try:
        games = retrieve_fn("video games")

        valid_games = (
            isinstance(games, list)
            and all(isinstance(game, dict) for game in games)
        )

        check(
            "Game retrieval",
            valid_games and len(games) > 0,
            f"Returned {len(games)} game records"
            if isinstance(games, list) else "Expected a list"
        )

        required_fields = {
            "Platform", "Name", "YearOfRelease", "Description"
        }

        fields_ok = (
            isinstance(games, list)
            and len(games) > 0
            and all(required_fields.issubset(game.keys()) for game in games)
        )

        check(
            "Retrieved game fields",
            fields_ok,
            "All required fields are present"
            if fields_ok else "Missing fields or no records returned"
        )

    except Exception as e:
        check("Game retrieval", False, f"{type(e).__name__}: {e}")


# ==================================================
# 3. Web search tool
# ==================================================
print("\n[3] WEB SEARCH")
print("-" * 55)

web_fn = globals().get("game_web_search")
tavily_key_present = bool(os.getenv("TAVILY_API_KEY"))

check(
    "game_web_search available",
    callable(web_fn),
    "Tool is defined" if callable(web_fn) else "Missing tool"
)

check(
    "Tavily API key configured",
    tavily_key_present,
    "Key is present" if tavily_key_present else "Key missing; load your .env"
)

if callable(web_fn) and tavily_key_present:
    try:
        web_result = web_fn(
            "When was the original Super Mario Bros. released?"
        )

        web_ok = (
            isinstance(web_result, str)
            and bool(web_result.strip())
            and "No relevant web search results were found." not in web_result
        )

        check(
            "Live web search",
            web_ok,
            "Non-empty search results returned"
            if web_ok else "No useful search results returned"
        )

    except Exception as e:
        check("Live web search", False, f"{type(e).__name__}: {e}")
else:
    print("INFO: Live web search test skipped.")


# ==================================================
# 4. State-machine prerequisites
# ==================================================
print("\n[4] STATE MACHINE")
print("-" * 55)

required_functions = [
    "retrieve_game",
    "evaluate_retrieval",
    "game_web_search",
    "generate_answer",
    "build_agent",
]

for name in required_functions:
    available = callable(globals().get(name))
    check(
        f"{name} defined",
        available,
        "Available" if available else "Define this function before testing"
    )


# ==================================================
# 5. Build and execute the graph
# ==================================================
print("\n[5] STATE-MACHINE EXECUTION")
print("-" * 55)

graph = None


def generate_answer(query, games, web_results):
    """
    Generate an answer using retrieved game records
    and optional web search results.

    Args:
        query: User's question.
        games: List of game records retrieved from ChromaDB.
        web_results: Web search results, if local evidence is insufficient.

    Returns:
        A string containing the final answer.
    """
    # TODO: Call your LLM using the correct course-library interface.
    # Include query, games, and web_results in the prompt.
    raise NotImplementedError(
        "Connect this function to your course LLM implementation."
    )

if all(callable(globals().get(name)) for name in required_functions):
    try:
        graph = build_agent(
            retrieve_game=retrieve_game,
            evaluate_retrieval=evaluate_retrieval,
            game_web_search=game_web_search,
            generate_answer=generate_answer,
        )

        check(
            "State-machine compilation",
            callable(getattr(graph, "invoke", None)),
            "Compiled graph exposes invoke()"
        )

    except Exception as e:
        check(
            "State-machine compilation",
            False,
            f"{type(e).__name__}: {e}"
        )
else:
    check(
        "State-machine compilation",
        False,
        "One or more required functions are missing"
    )

# ==================================================
# FINAL SUMMARY
# ==================================================
print("\n" + "=" * 55)
print(f"ADVANCED TEST RESULT: {checks_passed}/{checks_total} checks passed")

if checks_passed == checks_total:
    print("✓ PASS: All advanced validation checks passed!")
else:
    print(f"✗ FAIL: {checks_total - checks_passed} check(s) need attention.")

print("=" * 55)


UdaPlay — Advanced Options Validation

[1] LONG-TERM MEMORY
-------------------------------------------------------
✓ PASS: Memory functions available — init_memory, save_memory and load_memory
✓ PASS: Memory save and load — Retrieved 2 test messages

[2] LOCAL GAME RETRIEVAL
-------------------------------------------------------
✓ PASS: retrieve_game available — Tool is defined
✓ PASS: Game retrieval — Returned 5 game records
✓ PASS: Retrieved game fields — All required fields are present

[3] WEB SEARCH
-------------------------------------------------------
✓ PASS: game_web_search available — Tool is defined
✓ PASS: Tavily API key configured — Key is present
✓ PASS: Live web search — Non-empty search results returned

[4] STATE MACHINE
-------------------------------------------------------
✓ PASS: retrieve_game defined — Available
✓ PASS: evaluate_retrieval defined — Available
✓ PASS: game_web_search defined — Available
✓ PASS: generate_answer defined — Available
✓ PASS: build_age